In [1]:
import pandas as pd
import GEOparse
from itertools import accumulate

In [2]:
# liver data
liver_mat = pd.read_csv('GSE17522_series_matrix.txt', skiprows=77, delimiter='\t', index_col=0)
selected_rows_liver = [30, 31]
liver_bxd = pd.read_csv('GSE17522_series_matrix.txt', skiprows=lambda x: x not in selected_rows_liver, delimiter='\t')
gse_liver = GEOparse.get_GEO(filepath="GSE17522_family.soft")
liver_plat = gse_liver.gpls['GPL6466'].table.set_index('ID')

30-Aug-2023 13:45:36 INFO GEOparse - Parsing GSE17522_family.soft: 
30-Aug-2023 13:45:36 DEBUG GEOparse - DATABASE: GeoMiame
30-Aug-2023 13:45:36 DEBUG GEOparse - SERIES: GSE17522
30-Aug-2023 13:45:36 DEBUG GEOparse - PLATFORM: GPL6466
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436705
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436706
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436707
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436708
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436709
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436710
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436711
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436712
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436713
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436714
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436715
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436716
30-Aug-2023 13:45:37 DEBUG GEOparse - SAMPLE: GSM436717
30-Aug-2023 13:45:38 DEBUG GEOparse 

In [3]:
# myeloid data
myeloid_mat = pd.read_csv('GSE18067_series_matrix.txt', skiprows=69, delimiter='\t', index_col=0)
selected_rows_myeloid = [69,45]
myeloid_bxd = pd.read_csv('GSE18067_series_matrix.txt', skiprows=lambda x: x not in selected_rows_myeloid, delimiter='\t')
gse_myeloid = GEOparse.get_GEO(filepath="GSE18067_family.soft")
myeloid_plat = gse_myeloid.gpls['GPL6238'].table.set_index('ID')[['Symbol']]

30-Aug-2023 13:45:45 INFO GEOparse - Parsing GSE18067_family.soft: 
30-Aug-2023 13:45:45 DEBUG GEOparse - DATABASE: GeoMiame
30-Aug-2023 13:45:45 DEBUG GEOparse - SERIES: GSE18067
30-Aug-2023 13:45:45 DEBUG GEOparse - PLATFORM: GPL6238
30-Aug-2023 13:45:46 DEBUG GEOparse - SAMPLE: GSM451699
30-Aug-2023 13:45:46 DEBUG GEOparse - SAMPLE: GSM451700
30-Aug-2023 13:45:46 DEBUG GEOparse - SAMPLE: GSM451701
30-Aug-2023 13:45:47 DEBUG GEOparse - SAMPLE: GSM451702
30-Aug-2023 13:45:47 DEBUG GEOparse - SAMPLE: GSM451703
30-Aug-2023 13:45:47 DEBUG GEOparse - SAMPLE: GSM451704
30-Aug-2023 13:45:47 DEBUG GEOparse - SAMPLE: GSM451705
30-Aug-2023 13:45:47 DEBUG GEOparse - SAMPLE: GSM451706
30-Aug-2023 13:45:48 DEBUG GEOparse - SAMPLE: GSM451707
30-Aug-2023 13:45:48 DEBUG GEOparse - SAMPLE: GSM451708
30-Aug-2023 13:45:48 DEBUG GEOparse - SAMPLE: GSM451709
30-Aug-2023 13:45:48 DEBUG GEOparse - SAMPLE: GSM451710
30-Aug-2023 13:45:48 DEBUG GEOparse - SAMPLE: GSM451711
30-Aug-2023 13:45:48 DEBUG GEOparse 

In [4]:
GB_data_MRK = pd.read_csv('MRK_Sequence.rpt', delimiter='\t', usecols=[1, 10])
RefSeq_data_MRK = pd.read_csv('MRK_Sequence.rpt', delimiter='\t', usecols=[1, 11])
genes_data = pd.read_csv('MGI_Coordinates.Build37.rpt.txt', index_col=2, sep='\t')
genes_data = genes_data[genes_data['marker type'] == 'Gene']

In [5]:
GB_data = GB_data_MRK.assign(GB=GB_data_MRK["GenBank IDs"].str.split("|")).explode("GB").drop('GenBank IDs', axis=1)
GB_data = GB_data.set_index('GB')
RefSeq_data = RefSeq_data_MRK.assign(RefSeq=RefSeq_data_MRK["RefSeq transcript IDs"].str.split("|")).explode("RefSeq").drop('RefSeq transcript IDs', axis=1)
RefSeq_data = RefSeq_data.set_index('RefSeq')
RefSeq_dic = RefSeq_data['Marker Symbol'].to_dict()
GB_dic = GB_data['Marker Symbol'].to_dict()
symbol_dic = {**RefSeq_dic, **GB_dic}

In [6]:
# positions
genes_positions = genes_data[['representative genome chromosome', 'representative genome start', 'representative genome end']]
genes_positions.dropna(inplace=True)
genes_positions = genes_positions[genes_positions['representative genome chromosome'].apply(lambda x: x not in ['Y', 'MT'])]
genes_positions= genes_positions.astype(int)
genes_positions = genes_positions.sort_values(by='representative genome chromosome')

C:\Users\Lenovo\anaconda3\lib\site-packages\pandas\util\_decorators.py:311: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return func(*args, **kwargs)


In [7]:
def process_coloumns(exp, bxd_info, rem, mapping):
    new_exp = exp.copy()
    bxd_info.columns, bxd_info.iloc[0] = bxd_info.iloc[0].to_list(), bxd_info.columns
    bxd_info_after_filter = bxd_info.drop(rem, axis=1)
    bxd_info_after_filter = bxd_info_after_filter.T
    temp = bxd_info_after_filter[0].tolist()
    bxd_info_after_filter['BXD'] = mapping(temp)
    bxd_info_after_filter = bxd_info_after_filter[['BXD']]
    bxd_dict = bxd_info_after_filter['BXD'].to_dict()
    new_exp.columns = new_exp.columns.map(bxd_dict)
    return new_exp

def process_rows(exp, annotation):
    new_exp = exp.copy()
    new_exp.index = new_exp.index.map(annotation)
    new_exp = new_exp[new_exp.index.notnull()]
    return new_exp

def process_exp(exp, bxd, annotation, rem, mapping):
    processed_exp_cols = process_coloumns(exp, bxd, rem, mapping)
    processed_exp = process_rows(processed_exp_cols, annotation)
    return processed_exp

def mapping_liver(bxd_list):
    return [bxd.split('_')[1] for bxd in bxd_list]

def mapping_myeloid(bxd_list):
    return ['BXD' + bxd[bxd.find(':') + 2:] for bxd in bxd_list]

def myeloid_same_loci_avg(exp):
    bxd_list = exp.columns.tolist()
    bxdS = [bxd.partition('.')[0] for bxd in bxd_list]
    bxd_map = {bxd:bxd.partition('.')[0] for bxd in bxd_list}
    exp_map = exp.copy()
    exp_map.columns = exp_map.columns.map(bxd_map)
    exp_avg = pd.DataFrame(index=exp_map.index)
    for i in bxdS:
        exp_avg[i] = exp_map[i].mean(axis=1)
    return exp_avg

def filter_data(exp, quantile):
    var_q = (exp.var(axis=1).to_frame()).quantile(q=quantile)[0]
    max_q = (exp.max(axis=1).to_frame()).quantile(q=quantile)[0]
    exp_filtered = exp[(exp.max(axis=1) > max_q) & (exp.var(axis=1) > var_q)]
    gene_list = exp_filtered.index.unique().tolist()
    exp_filtered_avg = pd.DataFrame(columns=exp_filtered.columns)
    for gene in gene_list:
        exp_filtered_avg.loc[gene] = exp_filtered.filter(like=gene, axis=0).mean(axis=0)
    return exp_filtered_avg

def gen_filter(genotypes, bxd):
    genotypes = genotypes.sort_values(by=['Chr_Build37', 'Build37_position'])
    prev_snp = genotypes.loc[1]
    rem_snp = list()
    for i in range(2, genotypes.shape[0]):
        curr_snp = genotypes.loc[i]
        if curr_snp['Chr_Build37'] == prev_snp['Chr_Build37']:
            if curr_snp[bxd].equals(prev_snp[bxd]):
                rem_snp.append(curr_snp['Locus'])
        prev_snp = genotypes.loc[i]
    return genotypes[~genotypes['Locus'].isin(rem_snp)]

def positions(data, chr_pos, chr_37, pos_37):
    pos_df = data[data[chr_37] == 1].copy()
    for i in range(2, 21):
        chr_i = data[data[chr_37] == i].copy()
        for pos in pos_37:
            chr_i[pos] = chr_i[pos].apply(lambda x: x + chr_pos[i-2])
        pos_df = pd.concat([pos_df, chr_i], axis=0)
    return pos_df

In [8]:
liver_proc = process_exp(liver_mat, liver_bxd, liver_plat['GENE_SYMBOL'].to_dict(), '!Sample_geo_accession', mapping_liver)
liver_proc = liver_proc.filter(like='BXD', axis=1)

myeloid_proc = process_exp(myeloid_mat, myeloid_bxd, myeloid_plat.Symbol.to_dict(), 'ID_REF', mapping_myeloid)

In [9]:
y, liver_algn = genes_positions.align(liver_proc, join='inner', axis=0)
z, myeloid_algn = genes_positions.align(myeloid_proc, join='inner', axis=0) 


In [10]:
liver_av = liver_algn.groupby(by=liver_algn.columns, axis=1).mean()
liver_filt = filter_data(liver_av, 0.35)

myeloid_av = myeloid_same_loci_avg(myeloid_algn)
myeloid_filt = filter_data(myeloid_av, 0.35)

In [11]:
genotypes = pd.read_excel('genotypes.xls', index_col=0, skiprows=1)
phenotypes = pd.read_excel('phenotypes.xls', index_col=0)

In [12]:
phenotypes_out = phenotypes.loc[[1002, 1003]].dropna(axis=1)
genotypes_bxd = genotypes.filter(like='BXD').columns
genotypes[genotypes_bxd] = genotypes[genotypes_bxd].applymap(str.upper)

In [13]:
liver_bxd_phen = [col for col in genotypes.columns if 'BXD' in col and col in liver_filt.columns]
liver_gen = gen_filter(genotypes, liver_bxd_phen)

myeolid_bxd_phen = [col for col in genotypes.columns if 'BXD' in col and col in myeloid_filt.columns]
myeolid_gen = gen_filter(genotypes, myeolid_bxd_phen)

phenotype_bxd = [col for col in genotypes.columns if 'BXD' in col and col in phenotypes_out.columns]
genotypes_phen = gen_filter(genotypes, phenotype_bxd)

In [14]:
chr_pos = [195154279, 181755017, 159745316, 156860686, 151758149, 149588044, 144995196, 130127694, 124359700,130530862, 121973369, 120092757, 120883175, 125139656, 104073951, 98008968, 95294699, 90720763,61420004]
chr_acum_pos = list(accumulate(chr_pos))

In [15]:
liver_pos = positions(liver_gen, chr_acum_pos, 'Chr_Build37', ['Build37_position']) 
myeloid_pos = positions(myeolid_gen, chr_acum_pos, 'Chr_Build37', ['Build37_position']) 
gene_pos = positions(genes_positions, chr_acum_pos,'representative genome chromosome', ['representative genome start', 'representative genome end'])

In [ ]:
liver_pos.set_index('Locus').to_excel('preprocess_sub\liver_pos_preprocessed.xlsx')
liver_filt.to_excel('preprocess_sub\liver_exp_preprocessed.xlsx')
liver_gen.to_excel('preprocess_sub\liver_gen_preprocessed.xlsx')


myeolid_gen.set_index('Locus').to_excel('preprocess_sub\myeolid_gen_preprocessed.xlsx')
myeloid_filt.to_excel('preprocess_sub\myeloid_exp_preprocessed.xlsx')
myeloid_pos.set_index('Locus').to_excel('preprocess_sub\myeloid_pos_preprocessed.xlsx')

gene_pos.to_excel('preprocess_sub\gene_pos_preprocessed.xlsx')

genotypes_phen.set_index('Locus').to_excel('preprocess_sub\genotypes_phen_preprocessed.xlsx')
phenotypes_out.to_excel('preprocess_sub\phenotypes_preprocessed.xlsx')